# 03 · Majority-class baseline

The baseline always predicts the most common training label. Use the same data preparation, split and training-only CV settings as notebook 02. Keep the test set aside.

## Load and split data

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, make_scorer, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

root = Path.cwd()
if not (root / "data").is_dir():
    root = root.parent

df = pd.read_csv(root / "data/raw/online_shoppers_intention.csv").drop_duplicates()
X = df.drop(columns="Revenue")
X["Weekend"] = X["Weekend"].astype(int)
y = df["Revenue"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

## Baseline pipeline

The dummy ignores feature values. Keep preprocessing in the pipeline so later models can use the same setup.

In [2]:
numeric_features = [
    "Administrative", "Administrative_Duration", "Informational",
    "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "PageValues", "SpecialDay",
]
categorical_features = [
    "Month", "VisitorType", "OperatingSystems", "Browser", "Region", "TrafficType",
]

preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("weekend", "passthrough", ["Weekend"]),
])


baseline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent")),
])

## Cross-validation results

F1, precision and recall refer to purchases. Report each metric's mean and standard deviation across the five folds.

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_folds = list(cv.split(X_train, y_train))

scoring = {
    "f1": make_scorer(f1_score, pos_label=True, zero_division=0),
    "precision": make_scorer(precision_score, pos_label=True, zero_division=0),
    "recall": make_scorer(recall_score, pos_label=True, zero_division=0),
    "accuracy": "accuracy",
}

scores = cross_validate(
    baseline, X_train, y_train, cv=cv_folds, scoring=scoring, error_score="raise"
)

results = pd.DataFrame({
    "mean": [scores[f"test_{metric}"].mean() for metric in scoring],
    "std": [scores[f"test_{metric}"].std() for metric in scoring],
}, index=pd.Index(scoring, name="metric"))
display(results.style.format("{:.2%}"))

,mean,std
metric,,
f1,0.00%,0.00%
precision,0.00%,0.00%
recall,0.00%,0.00%
accuracy,84.37%,0.02%


## Findings

The baseline gets about 84.37% accuracy by predicting no purchase every time. Purchase F1, precision and recall are all zero: it finds none of the purchases. Later models need to improve purchase detection, not just accuracy.

## References

- [scikit-learn: cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html)
- [scikit-learn: metrics and scoring](https://scikit-learn.org/stable/modules/model_evaluation.html)